# Sine wave prediction using Physics Informed Long Short-Term Memory networks (PI-LSTM)

This code implements the PINT LSTM ([Code](https://github.com/KV-Park/PINT/blob/main/PINT%5BTraining%5D.ipynb) | [Paper](https://arxiv.org/pdf/2502.04018)) to predict and forecast on synthetic data (sine waves).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

from helper_functions import create_sequences, forecast_with_model, set_seed, train_models
from models import RNN, PhysicsInformedRNN, LSTM, PhysicsInformedLSTM, GRU, PhysicsInformedGRU

* input_size = 1 because we have one feature (value per time step).
* hidden_size is the size of the hidden state.

In [ ]:
# Key parameters:
SEED_VALUE = 2026

# Preprocessing
# Experiment 1: Simple sine wave
# SEQ_LENGTH = 20  # Number of past time steps to look at 
# PRED_LENGTH = 10  # Number of time steps to predict

# Experiment 2:
# seq_length = 20/63 * 365  gives relative proportion out of period (63 for working wave, 365 for climate wave) that is in seq length
# SEQ_LENGTH = 120  # Number of past time steps to look at 
# PRED_LENGTH = 15 # Number of time steps to predict

# Experiment 5: Simple sine wave
SEQ_LENGTH = 90  # Number of past time steps to look at 
PRED_LENGTH = 30  # Number of time steps to predict


# Model set up
input_size=1
output_size = PRED_LENGTH
hidden_size=64
num_layers=1
dropout=0
NUM_EPOCHS=100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
set_seed(SEED_VALUE)

## Make data

In [ ]:
# Experiment 1: Simple
# t = np.linspace(0, 100, 1000)
# freq = 1/(2*np.pi)
# amp = 1
# phase = 0
# data = amp * np.sin(2*(np.pi)*freq*t + phase)

# # Experiment 2: Matching climate data
# amp = 1
# freq = 1/365  # where 365 is the period of the data
# phase = 0
# max_time = 1575
# num_timesteps = max_time  # one data point per day
# t = np.linspace(0, max_time, num_timesteps)
# data = amp*np.sin(2*(np.pi)*freq*t + phase)

# Experiment 3: have same frequency as basic sine wave but same data length and sampling rate as climate data
# t = np.linspace(0, 1575, 1575)
# freq = 1/(2*np.pi*10)
# amp = 1
# phase = 0
# data = amp * np.sin(2*(np.pi)*freq*t + phase)


# freq_climate = 1/365
# data_climate = amp * np.sin(2*(np.pi)*freq_climate*t + phase)
# test_climate = data_climate[int(len(data_climate)*0.85):]
# test_t_climate = t[int(len(data_climate)*0.85):]

# okay yay that worked so like at what period does the cut off happen where it can't learn?
# See changing frequency notebook

# Experiment 5:
# If we have loads and loads of data can it learn the long term trend?

amp = 1
freq = 1/365  # where 365 is the period of the data
phase = 0
max_time = 1575*20
num_timesteps = max_time  # one data point per day
t = np.linspace(0, max_time, num_timesteps)
data = amp*np.sin(2*(np.pi)*freq*t + phase)



In [ ]:
train = data[:int(len(data)*0.7)]
train_t = t[:int(len(data)*0.7)]
val = data[int(len(data)*0.7):int(len(data)*0.85)]
val_t = t[int(len(data)*0.7):int(len(data)*0.85)]
test = data[int(len(data)*0.85):]
test_t = t[int(len(data)*0.85):]
preds_t = test_t[SEQ_LENGTH:]  # doesn't predict for the first SEQ_LENGTH values of test set, because this is the first sequence used

In [ ]:
fig = plt.figure()
plt.plot(t, data, c="#1589e8")
plt.xlabel("t")
plt.ylabel("sin(t)")
plt.show()

In [ ]:
plt.plot(train_t, train, label="Training", c="#1589e8")
plt.plot(val_t, val, label="Validation", c="#dc267f")
plt.plot(test_t, test, label="Test", c="#ffb000")
plt.xlabel("t")
plt.ylabel("sin(wt)")
plt.show()

In [ ]:
# Create sequences
X_train, y_train = create_sequences(train, SEQ_LENGTH, PRED_LENGTH)
X_val, y_val = create_sequences(val, SEQ_LENGTH, PRED_LENGTH)
X_test, y_test = create_sequences(test, SEQ_LENGTH, PRED_LENGTH)

# Convert arrays to PyTorch tensors
X_train = torch.Tensor(X_train).unsqueeze(-1) # Shape: (batch, seq, input_size)
y_train = torch.Tensor(y_train)
X_val = torch.Tensor(X_val).unsqueeze(-1) # Shape: (batch, seq, input_size)
y_val = torch.Tensor(y_val)
X_test = torch.Tensor(X_test).unsqueeze(-1)
y_test = torch.Tensor(y_test)

In [ ]:
plt.scatter(np.arange(SEQ_LENGTH), X_train[0])

In [ ]:
len_data = len(X_train) + len(X_val) + len(X_test)
print(f"Train:val:test ratio: {len(X_train)}:\
{len(X_val)}:{len(X_test)}")
print(f"Train:val:test ratio: \
{len(X_train)/len_data*100:.2f}%:\
{len(X_val)/len_data*100:.2f}%:\
{len(X_test)/len_data*100:.2f}%")

## Define and train models

In [ ]:
# Initialise models
omega = 2 * np.pi * freq
models = {
    "RNN": RNN(input_size, hidden_size, num_layers, output_size),
    "PI-RNN": PhysicsInformedRNN(input_size, hidden_size, num_layers, output_size, omega_init=omega),
    "LSTM": LSTM(input_size, hidden_size, num_layers, output_size),
    "PI-LSTM": PhysicsInformedLSTM(input_size, hidden_size, num_layers, output_size, omega_init=omega),
    "GRU": GRU(input_size, hidden_size, num_layers, output_size),
    "PI-GRU": PhysicsInformedGRU(input_size, hidden_size, num_layers, output_size, omega_init=omega)
}

In [ ]:
print("=== Starting training ===")
train_models(models, X_train, y_train, X_val, y_val, num_epochs=NUM_EPOCHS, physics_loss_weight=0.005)

## Evaluate
Check if predicted sequences align with true values


In [ ]:
colours = {  # IBM colourblind palette
    "Ground truth": "#1589e8",
    "LSTM": "#dc267f",
    "PI-LSTM": "#dc267f",
    "Starting data": "#631ff3",
    "RNN": "#fe5100",
    "PI-RNN": "#fe5100",
    "GRU": "#ffb000",
    "PI-GRU": "#ffb000"
}
    # "#631ff3", "#fe5100" # - IBM colourblind palette

In [ ]:
y_preds = {}
for name, model in models.items():
    model.eval()
    with torch.no_grad():
        preds = model(X_test).squeeze().numpy()
        y_preds[name] = preds # Predict on test set

In [ ]:
# Plot 5 random examples from test set!
for i in np.random.choice(range(len(y_test)), 5):
    # Plot actual vs predicted 
    plt.figure(figsize=(10, 4))
    times = preds_t[i:i+PRED_LENGTH]
    plt.plot(times, y_test[i], '--', label='Ground truth', c=colours['Ground truth'])
    for name in models.keys():
        ls = '--' if "PI" in name else '-'
        m = 'x' if "PI" in name else '.'
        plt.plot(times, y_preds[name][i], label=name, c=colours[name], marker=m, ls=ls)
    ax = plt.gca()
    ax.grid(visible=True, axis="x")
    plt.xticks(rotation=90)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.show()

In [ ]:
for name in models.keys():
    plt.figure(figsize=(10, 4))
    # y_test_inv = np.array([scaler.inverse_transform(test.reshape(-1, 1)) for test in y_test])
    # plt.plot(test_df["datetimes"].values[SEQ_LENGTH:], y_test_inv, label='Ground truth', c=colours['Ground truth'])
    plt.plot(preds_t, test[SEQ_LENGTH:], label='Ground truth', c=colours['Ground truth'])

    for i in range(len(y_test)):
        times = preds_t[i:i+PRED_LENGTH]
        plt.plot(times, y_preds[name][i], c=colours[name])

    ax = plt.gca()
    ax.grid(visible=True)
    plt.xticks(rotation=90, ha="right")
    plt.xlabel("Month")
    plt.ylabel("Mean Temperature (°C)")
    handles, _ = plt.gca().get_legend_handles_labels()
    line = Line2D([0], [0], label=name, c=colours[name])
    handles.extend([line])
    plt.legend(handles=handles, bbox_to_anchor=(1,1))
    plt.show()

In [ ]:
for name in models.keys():
    # Calculate RMSE
    rmse = [np.sqrt(np.mean((y_preds[name][i] - y_test[i].numpy()) ** 2)) for i in range(len(y_preds[name]))]
    
    # Calculate Pearson correlation
    correlation, _ = pearsonr(y_preds[name].squeeze(), np.array(y_test))

    print(f"Metrics for model - {name}:")
    print(f"RMSE:\t\t{np.mean(rmse)} +/- {np.std(rmse)}")
    print(f"P-CORR:\t\t{np.mean(correlation)} +/- {np.std(correlation)}")

In [ ]:
plt.figure(figsize=(6, 6))
for name in models.keys():
    plt.scatter(y_test.reshape(-1,1), y_preds[name].reshape(-1,1), alpha=0.5, label=name, c=colours[name])
plt.plot([y_test.min(), y_test.max()], 
         [y_test.min(), y_test.max()], 
         '--', lw=2, c=colours['Ground truth'])
plt.title('Correlation Plot (Actual vs Predicted)')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.grid(True)
plt.legend()
plt.show()

## Autoregressive Forecasting

In [ ]:
y_forecast = {}
start_data = X_test[0:1]  # Starting data: first sequence of X_test
desired_length = len(test)-SEQ_LENGTH
for name, model in models.items():
    print(f"Forcasting for model: {name}")
    y_forecast[name] = forecast_with_model(model, start_data, desired_length, PRED_LENGTH)

In [ ]:
for name in models.keys():
    # Calculate RMSE
    rmse = np.sqrt(np.mean((y_forecast[name].squeeze() - test[SEQ_LENGTH:]) ** 2))

    # Calculate Pearson correlation
    correlation, _ = pearsonr(y_forecast[name].squeeze(), test[SEQ_LENGTH:])

    print(f"Metrics for model - {name}:")
    print(f"RMSE:\t\t{rmse}")
    print(f"P-CORR:\t\t{correlation}")

In [ ]:
# Plot actual vs predicted
plt.figure(figsize=(10, 4))

# Starting data: first sequence of X_test
plt.plot(test_t[:SEQ_LENGTH], start_data.squeeze(), label='Starting data', c=colours['Starting data'])

# Ground truth: test data across region predicted for
plt.plot(preds_t[:1575], test[SEQ_LENGTH:1575+SEQ_LENGTH], label='Ground truth', c=colours['Ground truth'])

# Predictions
for name in models.keys():
    ls = '--' if "PI" in name else '-'
    m = 'x' if "PI" in name else '.'
    plt.plot(preds_t[:1575], y_forecast[name][:1575], label=name, c=colours[name], ls=ls)
plt.title("Autoregressive Forecasted Data")
ax = plt.gca()
plt.xticks(rotation=90)
plt.xlabel("t")
plt.ylabel("sin(wt)")
plt.legend()
plt.show()

In [ ]:
for type in ["RNN", "LSTM", "GRU"]:
    # Plot actual vs predicted
    plt.figure(figsize=(10, 4))

    # Starting data: first sequence of X_test
    plt.plot(test_t[:SEQ_LENGTH], start_data.squeeze(), label='Starting data', c=colours['Starting data'], lw=2)

    # Ground truth: test data across region predicted for
    plt.plot(preds_t, test[SEQ_LENGTH:], label='Ground truth', c=colours['Ground truth'], lw=2)

    # # Plot climate data for comparison
    # plt.plot(test_t_climate, test_climate, label='Climate frequency', c=colours['RNN'], lw=2, ls=':')

    # Predictions
    for name in [type, f"PI-{type}"]:
        # ls = '--' if "PI" in name else '-'
        c = colours["LSTM"] if "PI" in name else colours["GRU"]
        # m = 'x' if "PI" in name else '.'
        plt.plot(preds_t, y_forecast[name], label=name, c=c, alpha=0.8, lw=2)
    ax = plt.gca()
    plt.grid(True)
    plt.xticks(rotation=90)
    plt.xlabel("t")
    plt.ylabel("sin(t)")
    plt.legend()
    plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
for name in models.keys():
    m = 'x' if "PI" in name else '.'
    plt.scatter(test[SEQ_LENGTH:], y_forecast[name], alpha=0.5, label=name, c=colours[name], marker=m)
plt.plot([test[SEQ_LENGTH:].min(), test[SEQ_LENGTH:].max()], 
         [test[SEQ_LENGTH:].min(), test[SEQ_LENGTH:].max()], 
         '--', lw=2, c=colours['Ground truth'])
plt.title('Correlation Plot (Actual vs Predicted)')
plt.xlabel('Actual Temperature')
plt.ylabel('Predicted Temperature')
plt.grid(True)
plt.legend()
plt.show()
